[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankausberlin/deen-lupysta/blob/main/scripts/dash.ipynb)

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**OpenRouterModels**</font><br>
#@markdown * Uses **openrouter-cli**-pip to get model infos from OpenRouter. It **must be installed**: `uv tool install openrouter-cli` or `pip install openrouter-cli`
#@markdown * You must insert your **OpenRouter API-key** first with: `openrouter-cli configure`'.
#@markdown * You can **specify** a comma-separated **list of frontier models** to be **scanned**, whose **latest versions** will be displayed in the **overview**.
from ipywidgets import Button, Box, Layout, Textarea, VBox, HTML, HBox
import subprocess, json, pprint

class ORMButtonBox ():
  def __init__ (self, descriptions, clicker=None, maxchar=60, color=None):
    # remember stuff - list to set
    self.descriptions = descriptions if len (descriptions) == len (set (descriptions)) else set (descriptions)
    self.clicker, self.maxchar, self.color, self.position, self.button = clicker, maxchar, color, -1, None
    self._original_colors = {} # Dictionary to store original colors

    # make buttons
    self.buttons = [Button (description = i if len (i) <= maxchar else f'{i[:(maxchar-3)]}...',
                            layout      = Layout(width='auto', height='22px'),
                            tooltip     = f'{i}')
                    for i in descriptions]

    # put them in a box / bind event
    self.widget = Box (layout   = Layout (display='flex', flex_flow='wrap'),
                       children = self.buttons)
    for button in self.buttons:
      self._original_colors[button] = button.style.button_color # Store original color
      button.on_click (self._clicker)

  # An alternative for the Tab widget
  def _clicker (self, b):
    # search clicked button and remember
    self.position, self.button = [(i,but) for i, but in enumerate(self.buttons) if b == but][0]

    # unselect (color) all buttons and select new (list of colors here: https://www.quackit.com/css/css_color_codes.cfm)
    self.unselect()
    if self.color: self.buttons[self.position].style={'button_color':self.color}
    self.buttons[self.position].layout.border = '1px solid black'

    # fire event
    if self.clicker: self.clicker (self)

  def unselect (self):
    # unselect all buttons and restore original color
    for b in self.buttons:
      b.layout.border = ''
      b.style = {'button_color': self._original_colors.get(b, None)} # Restore original color, default to None if not found

def selectOrganization (bbox):
  global selectedOrga
  selectedOrga =  bbox.button.tooltip
  org_models = [m for m in json_list if m['id'].split('/')[0] == selectedOrga]
  modelsBox = ORMButtonBox ([i['id'].replace (bbox.button.tooltip+'/','') for i in org_models], clicker=selectModel, color='powderblue')
  widgetList.children=(widgetList.children[0], widgetList.children[1], modelsBox.widget)

def selectModel (bbox):
  for model in json_list:
    if model['id'] == selectedOrga+'/'+bbox.button.tooltip:
      if len (widgetList.children) < 4:
        widgetList.children = (*widgetList.children, Textarea (layout=Layout(width='auto', height='500px')))
      widgetList.children[3].value = pprint.pformat(model, width=160)

# ___________________________________________________________
#|______________________hello_component______________________|
try:
  json_list = json.loads(subprocess.run(['openrouter-cli','models', '--raw'], capture_output=True, text=True, check=True).stdout)
  tab_list = subprocess.run(['openrouter-cli','models'], capture_output=True, text=True, check=True).stdout.split('\n')
except: json_list = []

if json_list:

  # generate frontier models overview
  frontier_models = "gpt-5, claude-opus-4, claude-sonnet-4, gemini-3, deepseek-v4, minimax-m2, kimi-k2, glm-5, qwen3, mimo-v2, grok-4, mistral-medium-3-5"  #@param {type:"string"}
  ignore_frontier_models_with = "8b, 27b, 35b, a3b, -lite, -image, -micro, -nano, qwen3.6-flash"  #@param {type:"string"}
  ignore = ignore_frontier_models_with.replace(' ', '').split(',')

  # build the list with tuples ('versionnr', 'model-id') for the frontier models
  frontiers = frontier_models.replace(' ', '').split(',')

  # correct filter
  bad_id = lambda id: sum([f in id for f in ignore])

  frontier_version_id_list = [(j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0] , j['id'])
                              for f in frontiers for j in json_list
                                if f in j['id'] and
                                  ':free' not in j['id'] and
                                  j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0][-1] in '0123456789' and
                                  not bad_id (j['id'])]

  # build the frontier model overview
  tmp = []
  for frontier in frontiers:
    match = sorted ([f for f in frontier_version_id_list if frontier in f[1]])
    if not match: continue # catch empty lists due to rigorous filtering

    show_version = match[-1][0]
    versions = [f for f in match if show_version == f[0]]
    hm = f"<font color=blue size=3><b>{versions[0][1].split('/')[0]}</b></font><hr>"
    for version in versions:
      hm += f"<b>{version[1].split('/')[1]}</b><br>"
      idj = [j for j in json_list if j['id'] == version[1]][0]
      r, w  = f"{(float (idj['pricing']['prompt'])*1000000):.2f}", f"{(float (idj['pricing']['completion'])*1000000):.2f}"
      cr = f"{(float(idj['pricing']['input_cache_read'])*1000000):.2f}" if 'input_cache_read' in idj['pricing'] else ' '
      cw = f"{(float(idj['pricing']['input_cache_write'])*1000000):.2f}" if 'input_cache_write' in idj['pricing'] else ' '
      hm += f"Context: {idj['context_length']}<br>Price: {r} / {w}<br>[Cache: {cr} / {cw}]<br>"

    tmp.append (HTML(value=hm[:-4], layout={"border":"2px solid yellow"}))
  overview_panel = HBox (children=tmp)

  # build orga / model selector
  selectedOrga      = None
  organization_list = sorted (set ([j['id'].split('/')[0] for j in json_list if '/' in j['id']]))
  organizationsBox  = ORMButtonBox (organization_list, clicker=selectOrganization, color='powderblue')
  title             = HTML (value="<font color=green size=4><b>Overview Frontier Models Latest")
  widgetList        = VBox (children=[organizationsBox.widget,HTML(value='<hr>')])

  display (VBox (children=[title,overview_panel]), widgetList)